In [5]:
import json
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone

import duckdb
import requests

DB_PATH = "/Users/arnarfreyrerlingsson/Desktop/Polymarket_bitcoin_5min/db/polymarket.duckdb"
GAMMA_MARKETS = "https://gamma-api.polymarket.com/markets"
DATA_TRADES = "https://data-api.polymarket.com/trades"
PAGE = 1000          # max trades per request
MAX_OFFSET = 10000   # Data API refuses offsets above this
_local = threading.local()  # one requests.Session per worker thread
TMP_FILE = f"{tempfile.gettempdir()}/bitcoin_5m_raw_batch.json"

# API field -> (table column, type)
FIELDS = {
    "conditionId": ("CONDITION_ID", "VARCHAR"), "slug": ("SLUG", "VARCHAR"),
    "eventSlug": ("EVENT_SLUG", "VARCHAR"), "title": ("TITLE", "VARCHAR"), "icon": ("ICON", "VARCHAR"),
    "timestamp": ("TIMESTAMP", "BIGINT"), "side": ("SIDE", "VARCHAR"), "outcome": ("OUTCOME", "VARCHAR"),
    "outcomeIndex": ("OUTCOME_INDEX", "INTEGER"), "asset": ("ASSET", "VARCHAR"),
    "price": ("PRICE", "DOUBLE"), "size": ("SIZE", "DOUBLE"),
    "transactionHash": ("TRANSACTION_HASH", "VARCHAR"), "proxyWallet": ("PROXY_WALLET", "VARCHAR"),
    "name": ("NAME", "VARCHAR"), "pseudonym": ("PSEUDONYM", "VARCHAR"), "bio": ("BIO", "VARCHAR"),
    "profileImage": ("PROFILE_IMAGE", "VARCHAR"),
    "profileImageOptimized": ("PROFILE_IMAGE_OPTIMIZED", "VARCHAR"),
}
# Trades are written to a temp JSON file and bulk-loaded from it
# (passing rows as Python parameters is ~1000 rows/s, far too slow).
INSERT = (
    f"INSERT INTO bitcoin_5m_trades ({', '.join(col for col, _ in FIELDS.values())}) "
    f"SELECT {', '.join(f'to_timestamp({k})' if k == 'timestamp' else k for k in FIELDS)} "
    f"FROM read_json('{TMP_FILE}', format = 'array', columns = "
    f"{{{', '.join(f'{k}: {t!r}' for k, (_, t) in FIELDS.items())}}})"
)


def get_session():
    """Return this thread's requests.Session (Session isn't guaranteed thread-safe)."""
    if not hasattr(_local, "session"):
        _local.session = requests.Session()
    return _local.session


def get_json(url, params):
    """GET with retries; backs off on rate limits (429) and errors."""
    for attempt in range(8):
        try:
            resp = get_session().get(url, params=params, timeout=30)
            if resp.status_code == 429:
                raise requests.HTTPError("429 Too Many Requests")
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException:
            if attempt == 7:
                raise
            time.sleep(min(2 ** attempt, 30))


def day_markets(day):
    """Return [(slug, conditionId), ...] for the 288 five-minute markets of a UTC day."""
    first = int(day.timestamp())
    slugs = [f"btc-updown-5m-{first + k * 300}" for k in range(288)]
    markets = []
    for j in range(0, len(slugs), 50):
        batch = slugs[j:j + 50]
        params = [("closed", "true"), ("limit", len(batch))] + [("slug", s) for s in batch]
        markets += [(m["slug"], m["conditionId"]) for m in get_json(GAMMA_MARKETS, params)]
    return sorted(markets)


def market_trades(slug, condition_id):
    """Page through all taker trades of one market."""
    trades, offset = [], 0
    while offset <= MAX_OFFSET:
        page = get_json(DATA_TRADES, {"market": condition_id, "takerOnly": "true",
                                      "limit": PAGE, "offset": offset})
        trades += page
        if len(page) < PAGE:
            return trades
        offset += PAGE
    print(f"  WARNING {slug}: hit the API offset cap, only {len(trades)} trades loaded")
    return trades


def insert_trades(con, trades):
    """Bulk-insert trades."""
    if trades:
        with open(TMP_FILE, "w") as f:
            json.dump(trades, f)
        con.execute(INSERT)


def fmt(seconds):
    return str(timedelta(seconds=int(seconds)))

In [7]:
DATE_START = '2026-09-21'
DATE_END = '2026-09-21'
n_markets = 100
n_runners = 8     # parallel API requests


In [8]:
# Days are UTC. Each day has 288 markets; trades are fetched and stored n_markets markets at a time.
start = datetime.fromisoformat(DATE_START).replace(tzinfo=timezone.utc)
end = datetime.fromisoformat(DATE_END).replace(tzinfo=timezone.utc)
days = [start + timedelta(days=i) for i in range((end - start).days + 1)]

total_markets = len(days) * 288
done = 0
t0 = time.time()
con = duckdb.connect(DB_PATH)


first_slug = f"btc-updown-5m-{int(start.timestamp())}"
after_last_slug = f"btc-updown-5m-{int((end + timedelta(days=1)).timestamp())}"
deleted = con.execute("DELETE FROM bitcoin_5m_trades WHERE SLUG >= ? AND SLUG < ?",
                      [first_slug, after_last_slug]).fetchone()[0]
print(f"deleted {deleted} existing rows for {DATE_START} to {DATE_END}")

# Markets of a batch are fetched n_runners at a time in worker threads;
with ThreadPoolExecutor(n_runners) as pool:
    for i, day in enumerate(days, 1):
        markets = day_markets(day)

        for j in range(0, len(markets), n_markets):
            batch = markets[j:j + n_markets]
            trades = []
            for market in pool.map(lambda m: market_trades(*m), batch):
                trades += market
            insert_trades(con, trades)

            done += len(batch)
            elapsed = time.time() - t0
            eta = elapsed / done * (total_markets - done)
            print(f"[day {i}/{len(days)}] {day:%Y-%m-%d} | markets {j + len(batch)}/{len(markets)} | "
                  f"rows: {len(trades)} | elapsed {fmt(elapsed)} | ETA {fmt(eta)}")
    print(f"Day {i} completed")

con.close()

deleted 0 existing rows for 2026-08-02 to 2026-09-22
[day 1/52] 2026-08-02 | markets 100/288 | rows: 175659 | elapsed 0:00:21 | ETA 0:53:29
[day 1/52] 2026-08-02 | markets 200/288 | rows: 169132 | elapsed 0:00:39 | ETA 0:49:15
[day 1/52] 2026-08-02 | markets 288/288 | rows: 151786 | elapsed 0:00:55 | ETA 0:47:30
[day 2/52] 2026-08-03 | markets 100/288 | rows: 215734 | elapsed 0:01:22 | ETA 0:51:30
[day 2/52] 2026-08-03 | markets 200/288 | rows: 235170 | elapsed 0:01:43 | ETA 0:51:15
[day 2/52] 2026-08-03 | markets 288/288 | rows: 166486 | elapsed 0:02:11 | ETA 0:54:54
[day 3/52] 2026-08-04 | markets 100/288 | rows: 194229 | elapsed 0:02:30 | ETA 0:53:11
[day 3/52] 2026-08-04 | markets 200/288 | rows: 214149 | elapsed 0:02:51 | ETA 0:52:19
[day 3/52] 2026-08-04 | markets 288/288 | rows: 148287 | elapsed 0:03:12 | ETA 0:52:31
[day 4/52] 2026-08-05 | markets 100/287 | rows: 170050 | elapsed 0:03:31 | ETA 0:51:19
[day 4/52] 2026-08-05 | markets 200/287 | rows: 194984 | elapsed 0:03:52 | ET

In [9]:
con.close()